VSR

In [ ]:
from pylipextractor.lip_extractor import LipExtractor
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, GRU, Dropout, Bidirectional, TimeDistributed, Input


Preprocesamiento

In [ ]:
vocab = sorted(set("".join(transcripciones)))   # desde tus datos reales
num_clases = len(vocab) + 1                     # +1 por blank
char_to_idx = {c: i+1 for i, c in enumerate(vocab)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

In [ ]:
def encode(text):
    return [char_to_idx[c] for c in text]

LIP Detection

In [ ]:
#Mehrad Yaghoubi, PyLipExtractor, https://github.com/MehradYaghoubi/pylipextractor
#python -m pip install pylipextractor


'''video_path = "path/to/video.mp4"
    output_path = "output/lip_frames.npy"'''

def Extrae(video_path, output_path):
    LipExtractor.config.IMG_H = 50
    LipExtractor.config.IMG_W = 75
    LipExtractor.config.EMA_ALPHA = 0.3
    extractor = LipExtractor()
    frames, rtf = extractor.extract_lip_frames(video_path, output_path)
    if frames is not None:
        print(f"Extracted {len(frames)} lip frames (RTF={rtf})")
    return frames


CNN para Feature extraction 2D

In [ ]:
def CreaModelo(entrada):
    # entrada (T, 50, 75, 3)
    x = TimeDistributed(Conv2D(32, (3,3), activation='relu'))(entrada)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Conv2D(64, (3,3), activation='relu'))(x)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Conv2D(128, (3,3), activation='relu'))(x)
    x = TimeDistributed(MaxPooling2D((2,2)))(x)
    x = TimeDistributed(Flatten())(x)
    x = TimeDistributed(Dense(256, activation='relu'))(x)
    return x

GRU 

In [ ]:
def CreaGru(entrada, num_clases):
    x = Bidirectional(GRU(128, return_sequences=True))(entrada)
    salida = Dense(num_clases)(x)   
    return salida

Modelo

In [ ]:
def ModeloCompleto(num_clases, T=None):
    entrada = Input(shape=(T, 50, 75, 3))
    x = CreaModelo(entrada)          # CNN aplicada a cada fotograma
    salida = CreaGru(x, num_clases)     # BiGRU + Dense a num_clases
    return Model(entrada, salida)

modelo = ModeloCompleto(num_clases, T=None)
modelo.summary()
